## 1 · Modello locale

In [442]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain_core.tools import render_text_description

# Replace with 'qwen2.5-math:1.5b' for the math sub-agent if available
# Pulled from Ollama it must be: ollama pull qwen2.5:1.5b
def make_model(model_name: str = "qwen2.5:1.5b", temperature: float = 0.0):
    return ChatOllama(
        model=model_name,
        temperature=temperature,
        #reasoning=True,  
    )

base_model = make_model()
print("The model is ready")

The model is ready


## 2 · Tool del MathAgent

In [443]:
import math
import re

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression safely.
    Supports: +, -, *, /, %, **, sqrt(), log(), sin(), cos(), tan(), pi, e.
    Input example: 'sqrt(144) + 3**2'
    """
    allowed = {
        "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
        "sin": math.sin,   "cos": math.cos, "tan": math.tan,
        "pi": math.pi,     "e": math.e,
        "abs": abs,        "round": round,  "pow": pow,
        "factorial": math.factorial, "gcd": math.gcd,
        "comb": math.comb, "perm": math.perm,
    }
    sanitized = re.sub(r"[^0-9\+\-\*\/\%\(\)\.\,\s\_a-zA-Z]", "", expression)
    
    try:
        result = eval(sanitized, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as ex:
        return f"Error: {ex}"


@tool
def modular_arithmetic(a: int, b: int, mod: int, operation: str = "multiply") -> str:
    """
    Perform modular arithmetic: (a OP b) mod n.
    Operations: 'add', 'subtract', 'multiply', 'power'.
    Example: a=12, b=16, mod=24, operation='multiply'  →  (12*16) mod 24 = 0
    """
    ops = {
        "add":      (a + b) % mod,
        "subtract": (a - b) % mod,
        "multiply": (a * b) % mod,
        "power":    pow(a, b, mod),  # fast modular exponentiation
    }
    if operation not in ops:
        return f"Unknown operation '{operation}'. Use: add, subtract, multiply, power."
    return str(ops[operation])


@tool
def statistics_helper(data: str, operation: str) -> str:
    """
    Compute basic statistics on a comma-separated list of numbers.
    Operations: 'mean', 'median', 'std', 'variance', 'range', 'sort'.
    Input example: data='86,94,97,88,96', operation='median'
    """
    import statistics
    try:
        nums = [float(x.strip()) for x in data.split(",")]
    except ValueError:
        return "Error: provide comma-separated numbers."

    ops = {
        "mean":     lambda d: statistics.mean(d),
        "median":   lambda d: statistics.median(d),
        "std":      lambda d: statistics.stdev(d) if len(d) > 1 else 0,
        "variance": lambda d: statistics.variance(d) if len(d) > 1 else 0,
        "range":    lambda d: max(d) - min(d),
        "sort":     lambda d: sorted(d),
    }
    if operation not in ops:
        return f"Unknown operation. Use: {list(ops.keys())}"
    return str(ops[operation](nums))

print("Math tools loaded")

Math tools loaded


## 3 · Tool del KnowledgeAgent

In [444]:
@tool
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia and return the summary of the most relevant article.
    Use for factual recall: history, science, geography, famous people, etc.
    Input example: query='French Revolution'
    """
    import wikipedia
    try:
        wikipedia.set_lang("en")
        results = wikipedia.search(query, results=3)
        if not results:
            return "No Wikipedia results found."
        # Try pages in order until one works
        for title in results:
            try:
                page = wikipedia.page(title, auto_suggest=False)
                # Return first 800 chars of summary to stay within context
                print(f"TOOL[wikipedia_search] - wikipedia found: {page.title} - {page.summary[:50]}...")
                return page.summary[:800]
            except wikipedia.DisambiguationError as de:
                continue
        return "Could not retrieve a Wikipedia page."
    except Exception as ex:
        return f"Wikipedia error: {ex}"


@tool
def duckduckgo_search(query: str) -> str:
    """
    Search the web via DuckDuckGo and return top 3 text snippets.
    Use for recent facts or when Wikipedia is insufficient.
    Returns raw non-generated content (text snippets only — no LLM).
    """
    from duckduckgo_search import DDGS
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No results found."
        print(f"TOOL[duckduckgo_search] - found {len(results)} results for query: '{query}'")
        return "\n\n".join(
            f"[{r['title']}]\n{r['body']}" for r in results
        )
    except Exception as ex:
        return f"Search error: {ex}"

print("Knowledge tools loaded")

Knowledge tools loaded


## 4 · EXPERTS - Tools & System Prompts

In [445]:
# TOOLS
math_tools = [calculator, modular_arithmetic, statistics_helper]
knowledge_tools = [wikipedia_search, duckduckgo_search]

MATH_TOOLS_DESC = render_text_description(math_tools)
KNOWLEDGE_TOOLS_DESC = render_text_description(knowledge_tools)

# MATH
MATH_SYSTEM_PROMPT = f"""
You are the Math Specialist Agent for a quiz game.
YOU MUST SOLVE MATH PROBLEMS USING TOOLS. Do NOT attempt to calculate in your head.
Your ONLY job is to solve mathematical problems accurately.

You have access to the following tools:
{MATH_TOOLS_DESC}

RULES:
1. For ANY calculation (addition, roots, logarithms, etc.), you MUST call the appropriate tool. Do NOT attempt to calculate in your head.
2. If the problem requires multiple steps, use the tools one by one.
3. Once you have the final mathematical result, output it clearly.
"""

# KNOWLEDGE
KNOWLEDGE_SYSTEM_PROMPT = f"""
You are the Factual Knowledge Specialist Agent.
Your ONLY job is to verify facts, history, science, and general culture.
YOU MUST NEVER ANSWER FROM YOUR INTERNAL MEMORY. ALWAYS USE TOOLS TO VERIFY FACTS.

You have access to the following tools:
{KNOWLEDGE_TOOLS_DESC}

RULES:
1. Always search for the core subject of the question using your tools.
2. If Wikipedia fails, use DuckDuckGo.
3. Keep your final answer concise and factual. State the correct fact clearly.
"""

## 5 · Sub-agents

In [446]:
# LangGraph's create_react_agent handles the ReAct loop internally.

# ── MathAgent ────────────────────────────────────────────────────────────────
math_agent = create_agent(
    model=make_model(temperature=0.0),
    system_prompt=MATH_SYSTEM_PROMPT,
    tools=math_tools,
    debug=True,
)

# ── KnowledgeAgent ───────────────────────────────────────────────────────────
knowledge_agent = create_agent(
    model=make_model(temperature=0.0),
    system_prompt=KNOWLEDGE_SYSTEM_PROMPT,
    tools=knowledge_tools,
    debug=True,
)

print("Sub-agents created")


Sub-agents created


## 6 · Debug Agent calls

In [447]:
def print_agent_stats(result: dict, agent: str = "Orchestrator"):
    """
    Estrae e formatta in modo leggibile le statistiche e i metadati 
    dall'output di un agente LangChain/LangGraph.
    """
    # Estraiamo l'ultimo messaggio, che è l'AIMessage generato dal modello
    messages  = result.get("messages", [])
    ai_msg = messages[-1]
    human_msg = messages[0]  # Il messaggio umano (prompt)
    meta = getattr(ai_msg, "response_metadata", {})
    
    print("\n" + "="*40)
    print(f"LOG DELL'AGENTE - {agent} ({meta.get('model', 'Sconosciuto')})")
    print("="*40)
    
    #0. contenuto del prompt
    print(f"--- PROMPT ------\n{human_msg.content}\n------------")
    
    # 1. Risposta e Strumenti
    print(f"--- RAW ------\n {ai_msg.content}\n------------")
    
   
    # Tool calls: cercati in tutti gli AIMessage della conversazione
    tool_calls = []
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tool_calls.extend(msg.tool_calls)
            
    if tool_calls:
        print(f"TOOL usati ({len(tool_calls)}):")
        for tool in tool_calls:
            print(f"  - Nome: {tool.get('name')}")
            print(f"  - Argomenti: {tool.get('args')}")
    else:
        print("\nTOOL usati: Nessuno (Risoluzione diretta)")

    # 2. Consumo Token
    usage = getattr(ai_msg, "usage_metadata", {})
    if usage:
        print("\nCONSUMO TOKEN")
        print(f"  - Input (letti):     {usage.get('input_tokens', 0)}")
        print(f"  - Output (generati): {usage.get('output_tokens', 0)}")
        print(f"  - Totale:            {usage.get('total_tokens', 0)}")
        

    # 3. Tempi e Prestazioni (Convertiti da nanosecondi a secondi)
    if meta and "total_duration" in meta:
        total_sec = meta.get("total_duration", 0) / 1e9
        prompt_sec = meta.get("prompt_eval_duration", 0) / 1e9
        eval_sec = meta.get("eval_duration", 0) / 1e9
        
        print("\nPRESTAZIONI")
        print(f"  - Modello:           {meta.get('model', 'Sconosciuto')}")
        print(f"  - Lettura Prompt:    {prompt_sec:.2f} s")
        print(f"  - Generazione:       {eval_sec:.2f} s")
        print(f"  - Tempo Totale:      {total_sec:.2f} s")
    
    print("="*40 + "\n")

## 7 · Orchestrator

In [448]:
# ── Wrapping sub-agents come @tool ───────────────────────────────────────────

@tool
def ask_math_agent(question_with_choices: str) -> str:
    """
    Delegate a mathematical question to the Math specialist agent.
    Use for: arithmetic, algebra, calculus, statistics, probability,
    modular arithmetic, combinatorics, geometry.
    Pass the full question text and all answer choices.
    """
    result = math_agent.invoke(
        {"messages": [("human", question_with_choices)]},
        config={"recursion_limit": 12},
    )
    print_agent_stats(result, agent="MathAgent")
    return result["messages"][-1].content


@tool
def ask_knowledge_agent(question_with_choices: str) -> str:
    """
    Delegate a factual knowledge question to the Knowledge specialist agent.
    Use for: history, geography, science, literature, general culture,
    current events, famous people, definitions.
    Pass the full question text and all answer choices.
    """
    result = knowledge_agent.invoke(
        {"messages": [("human", question_with_choices)]},
        config={"recursion_limit": 8},
    )
    print_agent_stats(result, agent="KnowledgeAgent")
    return result["messages"][-1].content


# ── Orchestrator ──────────────────────────────────────────────────────────────
orchestrator_tools = [ask_math_agent, ask_knowledge_agent]
ORCHESTRATOR_TOOLS_DESC = render_text_description(orchestrator_tools)

ORCHESTRATOR_SYSTEM_PROMPT = f"""
You are the strict Routing Coordinator for a Quiz Game.
Your ONLY purpose is to read the user's question and immediately dispatch it to the correct specialist.
to call a tool use tool_calls[...]
Available Tools:
{ORCHESTRATOR_TOOLS_DESC}

CRITICAL RULES - READ CAREFULLY:
1. YOU ARE STRICTLY FORBIDDEN from answering the question yourself. Even if you know the answer, DO NOT answer it.
2. YOUR FIRST AND ONLY ACTION MUST BE TO CALL A TOOL. 
3. Choose 'ask_math_agent' for equations, calculations, derivatives, and math logic.
4. Choose 'ask_knowledge_agent' for science, history, biology, and factual trivia.
5. Pass the ENTIRE question and the 4 options (A,B,C,D) into the tool's argument.

Once the tool returns the observation, match it to the correct option and output ONLY the letter (A, B, C, or D).
"""

orchestrator = create_agent(
    model=make_model(temperature=0.0),
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
    tools=orchestrator_tools,
    debug=True,
)

print("Orchestrator ready")

Orchestrator ready


## 8 · Wrapper per il quiz

In [449]:
import re

def format_question(question: str, choices: dict) -> str:
    """Format question + choices for the orchestrator."""
    lines = [question, "Options:"]
    for letter, text in choices.items():
        lines.append(f"{letter}: {text}")
        
    
    lines.append("\n") #testo inseribile post risposta
    return "\n".join(lines)

def extract_letter(raw: str) -> str:
    """Extract A/B/C/D from agent output."""
    m = re.search(r"\b([A-Da-d])\b", raw.strip())
    return m.group(1).upper() if m else "A"

def answer_quiz_question(question: str, choices: dict) -> dict:
    """
    Entry point per il quiz.

    Parameters
    ----------
    question : str  — testo della domanda
    choices  : dict — {'A': '...', 'B': '...', 'C': '...', 'D': '...'}

    Returns
    -------
    {'answer_letter': 'B', 'answer_text': '...', 'raw': '...'}
    """
    prompt = format_question(question, choices)
    result = orchestrator.invoke(
        {"messages": [("human", prompt)]},
        config={"recursion_limit": 8},
    )
    print_agent_stats(result, agent="Orchestrator")
    
    raw    = result["messages"][-1].content
    letter = extract_letter(raw)
    return {
        "answer_letter": letter,
        "answer_text":   choices.get(letter, "?"),
        "raw":           raw,
    }




## 9 TEST

In [450]:
result = knowledge_agent.invoke(
    {"messages": [("human", "as a model what is the lastest train data used by you?")]},
    config={"recursion_limit": 4},
)
result["messages"][-1].content

[values] {'messages': [HumanMessage(content='as a model what is the lastest train data used by you?', additional_kwargs={}, response_metadata={}, id='2c14020f-fa8f-4c9f-8763-373bb7edf37e')]}
[updates] {'model': {'messages': [AIMessage(content="The latest training data I use was updated in October 2021. This includes information on over 5,000 languages and dialects, as well as their speakers and scripts. The dataset also contains details about language families, geographical distribution, and more. For the most current and accurate information, it's always a good idea to consult the latest sources or tools available.", additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-22T09:24:50.4463465Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7688893400, 'load_duration': 1548091000, 'prompt_eval_count': 442, 'prompt_eval_duration': 4395409300, 'eval_count': 79, 'eval_duration': 1410000300, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_pr

"The latest training data I use was updated in October 2021. This includes information on over 5,000 languages and dialects, as well as their speakers and scripts. The dataset also contains details about language families, geographical distribution, and more. For the most current and accurate information, it's always a good idea to consult the latest sources or tools available."

In [451]:
question="What is the units digit in the standard decimal expansion of 7^25?"
choices={"A": "5", "B": "7", "C": "3", "D": "1"} # B è la risposta corretta

result = answer_quiz_question(question, choices)
letter = extract_letter(result["answer_letter"])

print(f"ANSWER: {letter} → {choices.get(letter, '?')}")


[values] {'messages': [HumanMessage(content='What is the units digit in the standard decimal expansion of 7^25?\nOptions:\nA: 5\nB: 7\nC: 3\nD: 1\n\n', additional_kwargs={}, response_metadata={}, id='2b6a5406-9c3f-4ca8-82af-a1279c1b679d')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_math_agent", "arguments": {"question_with_choices": "What is the units digit in the standard decimal expansion of 7^25?"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-22T09:24:57.0793763Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6616944400, 'load_duration': 101348400, 'prompt_eval_count': 587, 'prompt_eval_duration': 5721321200, 'eval_count': 36, 'eval_duration': 642223400, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--019e4f00-749d-7c91-b1b7-9f72b333734a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 587, 'output_tokens': 36, 'total_tokens': 623})]}}
[

In [452]:
q_complex = "What is the result of rounding the square root of 948372 " \
    "multiplied by the base-10 logarithm of 55 to the nearest integer?"
    
c_complex = {
    "A": "1695",  # The correct answer
    "B": "1740",
    "C": "1582",
    "D": "974"
}

result = answer_quiz_question(q_complex, c_complex)
letter = extract_letter(result["answer_letter"])

print(f"ANSWER: {letter} → {c_complex.get(letter, '?')}")


[values] {'messages': [HumanMessage(content='What is the result of rounding the square root of 948372 multiplied by the base-10 logarithm of 55 to the nearest integer?\nOptions:\nA: 1695\nB: 1740\nC: 1582\nD: 974\n\n', additional_kwargs={}, response_metadata={}, id='099eabdd-a382-4df6-b810-c59916e082c5')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_math_agent", "arguments": {"question_with_choices": "What is the result of rounding the square root of 948372 multiplied by the base-10 logarithm of 55 to the nearest integer? Options:\\nA: 1695\\nB: 1740\\nC: 1582\\nD: 974"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-22T09:24:59.9287894Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2836383000, 'load_duration': 90672900, 'prompt_eval_count': 616, 'prompt_eval_duration': 766237200, 'eval_count': 87, 'eval_duration': 1532528000, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='l

In [453]:
# --- Test enciclopedico (Wikipedia) ---
q_wiki = (
    "Which ancient Greek philosopher was famously the personal tutor "
    "of Alexander the Great?"
)
c_wiki = {
    "A": "Socrates",
    "B": "Plato",
    "C": "Aristotle",  # Risposta corretta
    "D": "Pythagoras"
}

result = answer_quiz_question(q_wiki, c_wiki)
letter = extract_letter(result["answer_letter"])
print(f"ANSWER: {letter} → {c_wiki.get(letter, '?')}")

[values] {'messages': [HumanMessage(content='Which ancient Greek philosopher was famously the personal tutor of Alexander the Great?\nOptions:\nA: Socrates\nB: Plato\nC: Aristotle\nD: Pythagoras\n\n', additional_kwargs={}, response_metadata={}, id='7b7b9814-2d40-48cd-ae96-7b6fff38e40f')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_knowledge_agent", "arguments": {"question_with_choices": "Which ancient Greek philosopher was famously the personal tutor of Alexander the Great?"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-22T09:25:01.2498475Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1307307900, 'load_duration': 96018500, 'prompt_eval_count': 583, 'prompt_eval_duration': 435568000, 'eval_count': 34, 'eval_duration': 540790100, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--019e4f00-99a5-7be3-bda7-f11d484977f2-0', tool_calls=[], invalid_tool_calls=[], usage_metad

In [454]:
# --- Test di attualità (DuckDuckGo) ---
q_future = (
    "Who won the general classification of the Tour de France in the year 2025?"
)
c_future = {
    "A": "Jonas Vingegaard",
    "B": "Tadej Pogačar",  # Correct answer
    "C": "Remco Evenepoel",
    "D": "Primož Roglič"
}
result = answer_quiz_question(q_future, c_future)
letter = extract_letter(result["answer_letter"])
print(f"ANSWER: {letter} → {c_future.get(letter, '?')}")

[values] {'messages': [HumanMessage(content='Who won the general classification of the Tour de France in the year 2025?\nOptions:\nA: Jonas Vingegaard\nB: Tadej Pogačar\nC: Remco Evenepoel\nD: Primož Roglič\n\n', additional_kwargs={}, response_metadata={}, id='8c51b22c-73b2-4b53-98af-c1798da5fa5d')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_knowledge_agent", "arguments": {"question_with_choices": "Who won the general classification of the Tour de France in the year 2025?"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-22T09:25:02.8976221Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1626164600, 'load_duration': 94299000, 'prompt_eval_count': 605, 'prompt_eval_duration': 661538100, 'eval_count': 39, 'eval_duration': 702360500, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--019e4f00-9ed5-7c51-b04e-b99b278cec5f-0', tool_calls=[], invalid_tool_calls=[], usage_metada